In [ ]:
import os
import warnings

import basedosdados as bd
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

load_dotenv()

BILLING_PROJECT_ID = os.getenv("BASEDOSDADOS_BILLING_PROJECT_ID")
DATASET_ID = "br_sp_saopaulo_geosampa_iptu"
TABLE_ID = "iptu"
FULL_TABLE = f"basedosdados.{DATASET_ID}.{TABLE_ID}"

# IPTU / Cadastro Predial (São Paulo) — Análise Exploratória

Notebook de discovery da fonte de **custo (atributos por imóvel)** proposta em `02 - Data Sources/IPTU - Cadastro Predial (GeoSampa - SP).md`.

**Objetivos**
- Confirmar que a tabela é consultável e descobrir o schema real (nomes/tipos de coluna ainda não confirmados — a tabela expõe mais campos do que o subconjunto visto nos exemplos da própria Base dos Dados).
- Checar se os dados estão de fato escopados para São Paulo (mesma checagem que identificou o problema no notebook do Inside Airbnb).
- Avaliar se os campos de valor são um proxy usável de preço de mercado, e em qual granularidade geográfica os dados permitem cruzar com ITBI / Inside Airbnb.

## 1. Descoberta do schema (amostra pequena)

Primeiro uma amostra pequena — mais barato e rápido do que carregar a tabela inteira antes de sequer saber os nomes das colunas.

In [ ]:
query_sample = f"SELECT * FROM `{FULL_TABLE}` LIMIT 1000"
df_sample = bd.read_sql(query_sample, billing_project_id=BILLING_PROJECT_ID)

print(f"Linhas: {df_sample.shape[0]:,}  |  Colunas: {df_sample.shape[1]}")
print("\nColunas:")
print(df_sample.columns.tolist())
df_sample.head(3)

In [ ]:
def find_col(df: pd.DataFrame, *keywords: str) -> str | None:
    """Prioriza match exato do nome da coluna; senão, o primeiro match por substring.

    Uma busca por substring de uma palavra curta como "ano" pode colidir com uma
    coluna sem relação (ex.: "ano_construcao" em vez do campo de ano de referência
    do imposto) — checar o match exato primeiro evita escolher a coluna errada
    silenciosamente.
    """
    exact_candidates = {"_".join(keywords), "".join(keywords)}
    for col in df.columns:
        if col.lower() in exact_candidates:
            return col
    for col in df.columns:
        lowered = col.lower()
        if all(kw in lowered for kw in keywords):
            return col
    return None


col_valor_construcao = find_col(df_sample, "valor", "constr")
col_valor_terreno = find_col(df_sample, "valor", "terreno")
col_area_construida = find_col(df_sample, "area", "constr")
col_area_terreno = find_col(df_sample, "area", "terreno")
col_bairro = find_col(df_sample, "bairro") or find_col(df_sample, "distrito")
col_uso = find_col(df_sample, "uso")
col_ano = find_col(df_sample, "ano")
col_geom = find_col(df_sample, "centroide") or find_col(df_sample, "geometr")

print(f"valor construcao: {col_valor_construcao}")
print(f"valor terreno:     {col_valor_terreno}")
print(f"area construida:   {col_area_construida}")
print(f"area terreno:      {col_area_terreno}")
print(f"bairro/distrito:   {col_bairro}")
print(f"tipo de uso:       {col_uso}")
print(f"ano:               {col_ano}")
print(f"geometria:         {col_geom}")

assert col_ano is not None, (
    "Nenhuma coluna 'ano' encontrada na amostra do schema — cheque df_sample.columns "
    "manualmente e defina col_ano explicitamente antes de continuar."
)

## 2. Ano mais recente disponível

In [ ]:
query_years = f"SELECT DISTINCT {col_ano} AS ano, COUNT(*) AS n FROM `{FULL_TABLE}` GROUP BY ano ORDER BY ano DESC LIMIT 10"
df_years = bd.read_sql(query_years, billing_project_id=BILLING_PROJECT_ID)
print(df_years)

latest_year = int(df_years["ano"].max())
print(f"\nUsando o ano mais recente: {latest_year}")

## 3. Carregar o ano mais recente completo

A tabela completa cobre 1995–2025 (~85M linhas); filtrar por um ano mantém o volume em um tamanho que o pandas comporta bem (~2-3M linhas, estimativa a partir do total/30 anos). Ainda assim pode demorar — é uma consulta real via rede + BigQuery, não leitura de arquivo local.

In [ ]:
# Coloca o valor do filtro entre aspas se a coluna for STRING no BigQuery — comparar um
# literal numérico sem aspas com uma coluna STRING gera erro de tipo na consulta, não um
# resultado errado silencioso, mas vale evitar em vez de descobrir depois de uma falha.
year_is_string = pd.api.types.is_string_dtype(df_sample[col_ano])
year_literal = f"'{latest_year}'" if year_is_string else str(latest_year)

query_full = f"SELECT * FROM `{FULL_TABLE}` WHERE {col_ano} = {year_literal}"
df_iptu = bd.read_sql(query_full, billing_project_id=BILLING_PROJECT_ID)
print(f"Linhas: {df_iptu.shape[0]:,}  |  Colunas: {df_iptu.shape[1]}")
df_iptu.head(3)

## 4. Qualidade dos dados: valores ausentes

In [ ]:
missing_pct = (df_iptu.isna().mean() * 100).sort_values(ascending=False)
fully_empty = missing_pct[missing_pct == 100].index.tolist()
print(f"{len(fully_empty)} colunas 100% vazias:")
print(fully_empty)

fig, ax = plt.subplots(figsize=(10, 8))
missing_pct.head(25).sort_values().plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_xlabel("% ausente")
ax.set_title("Top 25 colunas por taxa de valores ausentes")
plt.tight_layout()
plt.show()

## 5. Checagem de escopo geográfico

Mesma checagem que identificou o problema São Paulo/Rio de Janeiro no notebook do Inside Airbnb — confirma se este cadastro está de fato escopado para São Paulo. `centroide` (se existir) vem do tipo GEOGRAPHY do BigQuery como texto WKT em WGS84 (EPSG:4326), sem necessidade de reprojeção.

In [ ]:
if col_bairro:
    print(f"Valores únicos de {col_bairro} ({df_iptu[col_bairro].nunique()}):")
    print(sorted(df_iptu[col_bairro].dropna().unique())[:40])
else:
    print("Nenhuma coluna de bairro/distrito encontrada — cheque a lista de colunas manualmente.")

if col_geom:
    geom = gpd.GeoSeries.from_wkt(df_iptu[col_geom].dropna(), crs="EPSG:4326")
    lon_min, lat_min, lon_max, lat_max = geom.total_bounds
    print(f"\nBounding box: lat [{lat_min:.3f}, {lat_max:.3f}], lon [{lon_min:.3f}, {lon_max:.3f}]")
    print("(São Paulo fica aproximadamente entre lat -23.9..-23.4, lon -46.9..-46.4)")

    fig, ax = plt.subplots(figsize=(8, 8))
    geom.sample(min(20_000, len(geom)), random_state=0).plot(ax=ax, markersize=0.5, color="#4C72B0")
    ax.set_title("Amostra dos centroides dos imóveis")
    plt.tight_layout()
    plt.show()
else:
    print("\nNenhuma coluna de geometria encontrada — cheque a lista de colunas manualmente.")

## 6. Limpeza dos campos numéricos

A Base dos Dados cura/tipa suas tabelas — os campos podem já vir numéricos. Checar o dtype antes de aplicar qualquer limpeza de string.

In [ ]:
def clean_numeric_br(series: pd.Series) -> pd.Series:
    """CSVs de órgãos públicos brasileiros costumam usar ',' como separador decimal e '.' como separador de milhar."""
    if pd.api.types.is_numeric_dtype(series):
        return series
    return (
        series.astype(str)
        .str.replace(r"[^0-9,.\-]", "", regex=True)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .replace("", np.nan)
        .astype(float)
    )


for col, label in (
    (col_valor_construcao, "valor_construcao_clean"),
    (col_valor_terreno, "valor_terreno_clean"),
    (col_area_construida, "area_construida_clean"),
    (col_area_terreno, "area_terreno_clean"),
):
    if col:
        df_iptu[label] = clean_numeric_br(df_iptu[col])
        print(f"{label} ({col}):")
        print(df_iptu[label].describe())
        print(f"Ausentes: {df_iptu[label].isna().mean():.1%}\n")
    else:
        print(f"Pulando {label} — coluna de origem não encontrada.\n")

if "valor_construcao_clean" in df_iptu and "valor_terreno_clean" in df_iptu:
    df_iptu["valor_venal_total"] = df_iptu["valor_construcao_clean"].fillna(0) + df_iptu["valor_terreno_clean"].fillna(0)
    print("valor_venal_total (construcao + terreno):")
    print(df_iptu["valor_venal_total"].describe())

## 7. Distribuição por tipo de uso

In [ ]:
if col_uso:
    fig, ax = plt.subplots(figsize=(10, 6))
    df_iptu[col_uso].value_counts().head(15).plot(kind="barh", ax=ax, color="#4C72B0")
    ax.set_title("Top 15 categorias de tipo de uso")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("Nenhuma coluna de tipo de uso encontrada — cheque a lista de colunas manualmente.")

## 8. Valor por região (bairro)

In [ ]:
value_col = "valor_venal_total" if "valor_venal_total" in df_iptu else "valor_construcao_clean"

if col_bairro and value_col in df_iptu:
    region_venal = (
        df_iptu.groupby(col_bairro)[value_col]
        .agg(median_valor="median", n_imoveis="count")
        .query("n_imoveis >= 100")
        .sort_values("median_valor", ascending=False)
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    region_venal.head(15)["median_valor"].sort_values().plot(kind="barh", ax=axes[0], color="#C44E52")
    axes[0].set_title("15 maiores valores medianos (n>=100)")

    region_venal.tail(15)["median_valor"].sort_values().plot(kind="barh", ax=axes[1], color="#4C72B0")
    axes[1].set_title("15 menores valores medianos (n>=100)")

    plt.tight_layout()
    plt.show()
else:
    print("Falta coluna de região ou de valor — não é possível segmentar por região ainda.")

## 9. Correlações

In [ ]:
numeric_cols = [
    c for c in (
        "valor_venal_total", "valor_construcao_clean", "valor_terreno_clean",
        "area_construida_clean", "area_terreno_clean",
    ) if c in df_iptu
]
if len(numeric_cols) >= 2:
    corr = df_iptu[numeric_cols].corr()
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
    ax.set_title("Correlação — valor venal vs. áreas")
    plt.tight_layout()
    plt.show()
else:
    print("Colunas numéricas limpas insuficientes para matriz de correlação.")